# BRFSS 2015 - split, scaling and resampling comparison

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Laiba-Gul/explainable-diabetes-prediction/blob/main/notebooks/02_brfss_preprocessing_and_resampling.ipynb)

**Status:** supporting research notebook (original Colab work, cleaned). Outputs were cleared to keep the repository small and free of personal metadata; re-run it or see the reproducible results in [`results/`](../results/).

**What it does:** Stratified 70/15/15 split, train-only scaling of ordinal/continuous features, comparison of SMOTE, Borderline-SMOTE, SMOTE-Tomek and SMOTE-ENN applied to training data only, feature-scoring (MI, ANOVA, information gain, gain ratio).

**Data:** run `diabetes-xai download` (or upload the CSV when the notebook asks). The CSVs are checksum-verified and are not stored in Git.


In [ ]:
# ==========================================
# Import Libraries
# ==========================================

import warnings
warnings.filterwarnings("ignore")

import os
import joblib
import numpy as np
import pandas as pd

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# ==========================================
# Create Required Folders
# ==========================================

folders = [

    "Processed_Data",
    "Processed_Data/Original",
    "Processed_Data/Scaled",

    "Models",

    "Results",

    "Results/Tables",
    "Results/Figures",

    "Scaler"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folders Created Successfully.")

In [ ]:
uploaded = files.upload()

In [ ]:
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

print("Dataset Loaded Successfully")

df.head()

In [ ]:
target = "Diabetes_binary"

print(df[target].value_counts())

print()

print(df[target].value_counts(normalize=True)*100)

In [ ]:
print(df.isnull().sum().sum())

In [ ]:
# ==========================================
# Separate Features and Target
# ==========================================

TARGET = "Diabetes_binary"

X = df.drop(columns=[TARGET])
y = df[TARGET]

print(f"Feature Matrix Shape : {X.shape}")
print(f"Target Vector Shape  : {y.shape}")

In [ ]:
from sklearn.model_selection import train_test_split

# ------------------------------------------
# First Split: 70% Train | 30% Temp
# ------------------------------------------

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# ------------------------------------------
# Second Split:
# 30% → 15% Validation + 15% Test
# ------------------------------------------

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("="*50)
print("Dataset Split")
print("="*50)

print("Training   :", X_train.shape)
print("Validation :", X_val.shape)
print("Testing    :", X_test.shape)

In [ ]:
def show_distribution(name, labels):
    dist = labels.value_counts(normalize=True).sort_index() * 100

    print(f"\n{name}")
    print("-"*25)

    for cls, pct in dist.items():
        print(f"Class {cls}: {pct:.2f}%")

show_distribution("Training Set", y_train)
show_distribution("Validation Set", y_val)
show_distribution("Testing Set", y_test)

In [ ]:
import os

os.makedirs("Processed_Data/Original", exist_ok=True)

X_train.to_csv("Processed_Data/Original/X_train.csv", index=False)
X_val.to_csv("Processed_Data/Original/X_val.csv", index=False)
X_test.to_csv("Processed_Data/Original/X_test.csv", index=False)

y_train.to_csv("Processed_Data/Original/y_train.csv", index=False)
y_val.to_csv("Processed_Data/Original/y_val.csv", index=False)
y_test.to_csv("Processed_Data/Original/y_test.csv", index=False)

print("✅ Original train, validation, and test splits saved.")

In [ ]:
# ==========================================
# Save Original Train, Validation and Test Sets
# ==========================================

import os

os.makedirs("Processed_Data/Original", exist_ok=True)

X_train.to_csv("Processed_Data/Original/X_train.csv", index=False)
X_val.to_csv("Processed_Data/Original/X_val.csv", index=False)
X_test.to_csv("Processed_Data/Original/X_test.csv", index=False)

y_train.to_csv("Processed_Data/Original/y_train.csv", index=False)
y_val.to_csv("Processed_Data/Original/y_val.csv", index=False)
y_test.to_csv("Processed_Data/Original/y_test.csv", index=False)

print("Original datasets saved successfully.")

In [ ]:
split_info = pd.DataFrame({

    "Dataset":[
        "Training",
        "Validation",
        "Testing"
    ],

    "Samples":[
        len(X_train),
        len(X_val),
        len(X_test)
    ],

    "Percentage":[
        70,
        15,
        15
    ]

})

split_info

In [ ]:
split_info.to_csv(
    "Results/Tables/dataset_split_information.csv",
    index=False
)

**Identify Feature Types**

We are not scaling everything.

In [ ]:
binary_features = [

    "HighBP",
    "HighChol",
    "CholCheck",
    "Smoker",
    "Stroke",
    "HeartDiseaseorAttack",
    "PhysActivity",
    "Fruits",
    "Veggies",
    "HvyAlcoholConsump",
    "AnyHealthcare",
    "NoDocbcCost",
    "DiffWalk",
    "Sex"

]

ordinal_features = [

    "GenHlth",
    "Age",
    "Education",
    "Income"

]

continuous_features = [

    "BMI",
    "MentHlth",
    "PhysHlth"

]

print("Binary Features :", len(binary_features))
print("Ordinal Features :", len(ordinal_features))
print("Continuous Features :", len(continuous_features))

In [ ]:
feature_types = pd.DataFrame({

    "Feature":

        binary_features
        + ordinal_features
        + continuous_features,

    "Type":

        ["Binary"]*len(binary_features)

        + ["Ordinal"]*len(ordinal_features)

        + ["Continuous"]*len(continuous_features)

})

feature_types

In [ ]:
feature_types.to_csv(

    "Results/Tables/feature_types.csv",

    index=False

)

In [ ]:
# ==========================================
# Standardize Continuous and Ordinal Features
# ==========================================

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scale_features = ordinal_features + continuous_features

# Fit ONLY on training data
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()

X_train_scaled[scale_features] = scaler.fit_transform(
    X_train[scale_features]
)

X_val_scaled[scale_features] = scaler.transform(
    X_val[scale_features]
)

X_test_scaled[scale_features] = scaler.transform(
    X_test[scale_features])

print("Scaling completed successfully.")

In [ ]:
summary = pd.DataFrame({

    "Mean": X_train_scaled[scale_features].mean(),

    "Std": X_train_scaled[scale_features].std()

})

summary.round(4)

In [ ]:
summary.to_csv(
    "Results/Tables/scaling_summary.csv"
)

In [ ]:
summary = pd.DataFrame({

    "Mean": X_train_scaled[scale_features].mean(),

    "Std": X_train_scaled[scale_features].std()

})

summary.round(4)

In [ ]:
summary.to_csv(
    "Results/Tables/scaling_summary.csv"
)

In [ ]:
import joblib
import os

os.makedirs("Scaler", exist_ok=True)

joblib.dump(
    scaler,
    "Scaler/standard_scaler.pkl"
)

print("Scaler saved successfully.")

In [ ]:
os.makedirs("Processed_Data/Scaled", exist_ok=True)

X_train_scaled.to_csv(
    "Processed_Data/Scaled/X_train_scaled.csv",
    index=False
)

X_val_scaled.to_csv(
    "Processed_Data/Scaled/X_val_scaled.csv",
    index=False
)

X_test_scaled.to_csv(
    "Processed_Data/Scaled/X_test_scaled.csv",
    index=False
)

print("Scaled datasets saved successfully.")

Install imbalanced-learn (Colab)

In [ ]:
!pip install -q imbalanced-learn

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import BorderlineSMOTE
from imblearn.combine import SMOTEENN
from imblearn.combine import SMOTETomek

import os

os.makedirs("Processed_Data/Resampled", exist_ok=True)

In [ ]:
# ==========================================
# SMOTE
# ==========================================

smote = SMOTE(
    random_state=42
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_scaled,
    y_train
)

print("SMOTE Completed")

print(y_train_smote.value_counts())

In [ ]:
X_train_smote.to_csv(
    "Processed_Data/Resampled/X_train_SMOTE.csv",
    index=False
)

y_train_smote.to_csv(
    "Processed_Data/Resampled/y_train_SMOTE.csv",
    index=False
)

Section 14 – Apply Borderline-SMOTE

In [ ]:
border = BorderlineSMOTE(
    random_state=42
)

X_train_border, y_train_border = border.fit_resample(
    X_train_scaled,
    y_train
)

print("Borderline-SMOTE Completed")

print(y_train_border.value_counts())

In [ ]:
X_train_border.to_csv(
    "Processed_Data/Resampled/X_train_BorderlineSMOTE.csv",
    index=False
)

y_train_border.to_csv(
    "Processed_Data/Resampled/y_train_BorderlineSMOTE.csv",
    index=False
)

Section 15 – Apply SMOTE + Tomek

In [ ]:
smote_tomek = SMOTETomek(
    random_state=42
)

X_train_tomek, y_train_tomek = smote_tomek.fit_resample(
    X_train_scaled,
    y_train
)

print("SMOTE + Tomek Completed")

print(y_train_tomek.value_counts())

In [ ]:
X_train_tomek.to_csv(
    "Processed_Data/Resampled/X_train_SMOTETomek.csv",
    index=False
)

y_train_tomek.to_csv(
    "Processed_Data/Resampled/y_train_SMOTETomek.csv",
    index=False
)


Section 16 Apply SMOTE + ENN

In [ ]:
smote_enn = SMOTEENN(
    random_state=42
)

X_train_enn, y_train_enn = smote_enn.fit_resample(
    X_train_scaled,
    y_train
)

print("SMOTE + ENN Completed")

print(y_train_enn.value_counts())

In [ ]:
X_train_enn.to_csv(
    "Processed_Data/Resampled/X_train_SMOTEENN.csv",
    index=False
)

y_train_enn.to_csv(
    "Processed_Data/Resampled/y_train_SMOTEENN.csv",
    index=False
)

Section 17 – Compare Resampling Results

In [ ]:
comparison = pd.DataFrame({

    "Original": y_train.value_counts().sort_index().values,

    "SMOTE": y_train_smote.value_counts().sort_index().values,

    "BorderlineSMOTE": y_train_border.value_counts().sort_index().values,

    "SMOTE+Tomek": y_train_tomek.value_counts().sort_index().values,

    "SMOTE+ENN": y_train_enn.value_counts().sort_index().values

},

index=["Class 0","Class 1"])

comparison

In [ ]:
comparison.reset_index(inplace=True)
comparison.rename(columns={"index": "Class"}, inplace=True)

comparison.to_csv(
    "Results/Tables/resampling_comparison.csv",
    index=False
)

print("Resampling comparison saved successfully.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

methods = ["Original", "SMOTE", "BorderlineSMOTE", "SMOTE+Tomek", "SMOTE+ENN"]

class0 = [
    152834,
    152834,
    152834,
    151941,
    101388
]

class1 = [
    24742,
    152834,
    152834,
    151941,
    139157
]

x = np.arange(len(methods))
width = 0.35

plt.figure(figsize=(10,6))

plt.bar(
    x - width/2,
    class0,
    width,
    label="Class 0",
    color="#A8DADC"
)

plt.bar(
    x + width/2,
    class1,
    width,
    label="Class 1",
    color="#F4A261"
)

plt.xticks(x, methods, rotation=15)
plt.ylabel("Number of Samples")
plt.title("Comparison of Resampling Techniques", fontsize=15, fontweight="bold")
plt.legend()

plt.tight_layout()

plt.savefig(
    "Results/Figures/resampling_comparison.png",
    dpi=300
)

plt.show()

In [ ]:
processed_files = pd.DataFrame({

    "Dataset":[
        "Original",
        "SMOTE",
        "BorderlineSMOTE",
        "SMOTE+Tomek",
        "SMOTE+ENN"
    ],

    "Feature File":[
        "X_train.csv",
        "X_train_SMOTE.csv",
        "X_train_BorderlineSMOTE.csv",
        "X_train_SMOTETomek.csv",
        "X_train_SMOTEENN.csv"
    ],

    "Target File":[
        "y_train.csv",
        "y_train_SMOTE.csv",
        "y_train_BorderlineSMOTE.csv",
        "y_train_SMOTETomek.csv",
        "y_train_SMOTEENN.csv"
    ]

})

processed_files.to_csv(
    "Processed_Data/processed_dataset_reference.csv",
    index=False
)

processed_files

Compare Resampling Techniques

In [ ]:
# ==========================================
# Import Libraries for Resampling Comparison
# ==========================================

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

In [ ]:
# ==========================================
# Evaluation Function
# ==========================================

def evaluate_model(model, X_train, y_train, X_val, y_val):

    model.fit(X_train, y_train)

    predictions = model.predict(X_val)

    probabilities = model.predict_proba(X_val)[:,1]

    results = {

        "Accuracy": accuracy_score(y_val, predictions),

        "Precision": precision_score(y_val, predictions),

        "Recall": recall_score(y_val, predictions),

        "F1-score": f1_score(y_val, predictions),

        "ROC-AUC": roc_auc_score(y_val, probabilities),

        "PR-AUC": average_precision_score(y_val, probabilities)

    }

    return results

In [ ]:
# ==========================================
# Baseline Models
# ==========================================

logistic = LogisticRegression(
    random_state=42,
    max_iter=1000
)

random_forest = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

In [ ]:
# ==========================================
# Original Dataset
# ==========================================

lr_original = evaluate_model(

    logistic,

    X_train_scaled,
    y_train,

    X_val_scaled,
    y_val
)

rf_original = evaluate_model(

    random_forest,

    X_train,
    y_train,

    X_val,
    y_val
)

In [ ]:
# ==========================================
# Evaluate SMOTE
# ==========================================

lr_smote = evaluate_model(

    logistic,

    X_train_smote,
    y_train_smote,

    X_val_scaled,
    y_val
)

rf_smote = evaluate_model(

    random_forest,

    X_train_smote,
    y_train_smote,

    X_val_scaled,
    y_val
)

In [ ]:
# ==========================================
# Borderline-SMOTE
# ==========================================

lr_border = evaluate_model(

    logistic,

    X_train_border,
    y_train_border,

    X_val_scaled,
    y_val
)

rf_border = evaluate_model(

    random_forest,

    X_train_border,
    y_train_border,

    X_val_scaled,
    y_val
)

In [ ]:
# ==========================================
# SMOTE + Tomek
# ==========================================

lr_tomek = evaluate_model(

    logistic,

    X_train_tomek,
    y_train_tomek,

    X_val_scaled,
    y_val
)

rf_tomek = evaluate_model(

    random_forest,

    X_train_tomek,
    y_train_tomek,

    X_val_scaled,
    y_val
)

In [ ]:
# ==========================================
# SMOTE + ENN
# ==========================================

lr_enn = evaluate_model(

    logistic,

    X_train_enn,
    y_train_enn,

    X_val_scaled,
    y_val
)

rf_enn = evaluate_model(

    random_forest,

    X_train_enn,
    y_train_enn,

    X_val_scaled,
    y_val
)

In [ ]:
# ==========================================
# Mutual Information (Information Gain)
# ==========================================

from sklearn.feature_selection import mutual_info_classif

mi_scores = mutual_info_classif(
    X_train_smote,
    y_train_smote,
    random_state=42
)

mi_df = pd.DataFrame({
    "Feature": X_train_smote.columns,
    "Mutual_Information": mi_scores
})

mi_df = mi_df.sort_values(
    by="Mutual_Information",
    ascending=False
)

mi_df.reset_index(drop=True, inplace=True)

mi_df

In [ ]:
mi_df.to_csv(
    "Results/Tables/mutual_information_scores.csv",
    index=False
)

print("Mutual Information scores saved.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,7))

sns.barplot(
    data=mi_df,
    x="Mutual_Information",
    y="Feature",
    color="#A8DADC"
)

plt.title(
    "Mutual Information Feature Importance",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Score")
plt.ylabel("Feature")

plt.tight_layout()

plt.savefig(
    "Results/Figures/mutual_information.png",
    dpi=300
)

plt.show()

# ==========================================
# ANOVA F-test
# ==========================================

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif

ig_selector = SelectKBest(score_func=f_classif, k='all')

ig_selector.fit(X_train_smote, y_train_smote)

ig_scores = pd.DataFrame({
    "Feature": X_train_smote.columns,
    "ANOVA F-test": ig_selector.scores_
})

ig_scores = ig_scores.sort_values(
    by="Information_Gain",
    ascending=False
)

ig_scores.reset_index(drop=True, inplace=True)

ig_scores

In [ ]:
from sklearn.metrics import mutual_info_score
import numpy as np

gain_ratio = []

for feature in X_train_smote.columns:

    feature_values = X_train_smote[feature]

    ig = mutual_info_score(
        pd.qcut(feature_values, q=10, duplicates='drop'),
        y_train_smote
    )

    intrinsic_value = mutual_info_score(
        pd.qcut(feature_values, q=10, duplicates='drop'),
        pd.qcut(feature_values, q=10, duplicates='drop')
    )

    ratio = ig / intrinsic_value if intrinsic_value != 0 else 0

    gain_ratio.append(ratio)

gain_ratio_df = pd.DataFrame({

    "Feature": X_train_smote.columns,

    "Gain_Ratio": gain_ratio

})

gain_ratio_df = gain_ratio_df.sort_values(
    by="Gain_Ratio",
    ascending=False
)

gain_ratio_df.reset_index(drop=True, inplace=True)

gain_ratio_df

In [ ]:
!pip install -q info-gain

In [ ]:
from info_gain import info_gain
import pandas as pd

In [ ]:
# ==========================================
# Information Gain
# ==========================================

ig_scores = []

for feature in X_train_smote.columns:

    score = info_gain.info_gain(
        X_train_smote[feature],
        y_train_smote
    )

    ig_scores.append(score)

information_gain_df = pd.DataFrame({

    "Feature": X_train_smote.columns,
    "Information_Gain": ig_scores

})

information_gain_df = information_gain_df.sort_values(
    by="Information_Gain",
    ascending=False
).reset_index(drop=True)

information_gain_df

In [ ]:
information_gain_df.to_csv(
    "Results/Tables/information_gain_scores.csv",
    index=False
)

print("Information Gain scores saved successfully.")

In [ ]:
# ==========================================
# Gain Ratio
# ==========================================

import numpy as np
import pandas as pd

gain_ratio_scores = []

for feature in X_train_smote.columns:

    # Create bins for continuous variables
    feature_bins = pd.qcut(
        X_train_smote[feature],
        q=10,
        duplicates="drop"
    )

    # Split Information
    probabilities = feature_bins.value_counts(normalize=True)

    split_info = -np.sum(
        probabilities * np.log2(probabilities)
    )

    # Information Gain
    ig = info_gain.info_gain(
        X_train_smote[feature],
        y_train_smote
    )

    # Gain Ratio
    if split_info == 0:
        gain_ratio = 0
    else:
        gain_ratio = ig / split_info

    gain_ratio_scores.append(gain_ratio)

gain_ratio_df = pd.DataFrame({

    "Feature": X_train_smote.columns,

    "Gain_Ratio": gain_ratio_scores

})

gain_ratio_df = gain_ratio_df.sort_values(
    by="Gain_Ratio",
    ascending=False
).reset_index(drop=True)

gain_ratio_df

In [ ]:
gain_ratio_df.to_csv(
    "Results/Tables/gain_ratio_scores.csv",
    index=False
)

print("Gain Ratio scores saved successfully.")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,7))

plt.barh(
    gain_ratio_df["Feature"],
    gain_ratio_df["Gain_Ratio"],
    color="#B8E0D2"
)

plt.gca().invert_yaxis()

plt.title(
    "Gain Ratio Feature Ranking",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Gain Ratio")
plt.ylabel("Features")

plt.tight_layout()

plt.savefig(
    "Results/Figures/gain_ratio.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# Random Forest Feature Importance
# ==========================================

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_smote, y_train_smote)

rf_importance = pd.DataFrame({

    "Feature": X_train_smote.columns,

    "RF_Importance": rf.feature_importances_

})

rf_importance = rf_importance.sort_values(
    by="RF_Importance",
    ascending=False
).reset_index(drop=True)

rf_importance

In [ ]:
rf_importance.to_csv(
    "Results/Tables/random_forest_feature_importance.csv",
    index=False
)

In [ ]:
plt.figure(figsize=(10,7))

plt.barh(
    rf_importance["Feature"],
    rf_importance["RF_Importance"],
    color="#A8DADC"
)

plt.gca().invert_yaxis()

plt.title(
    "Random Forest Feature Importance",
    fontsize=15,
    fontweight="bold"
)

plt.tight_layout()

plt.savefig(
    "Results/Figures/random_forest_feature_importance.png",
    dpi=300
)

plt.show()

In [ ]:
!pip install -q xgboost

In [ ]:
from xgboost import XGBClassifier

In [ ]:
xgb = XGBClassifier(

    n_estimators=300,

    learning_rate=0.05,

    max_depth=6,

    random_state=42,

    eval_metric="logloss"

)

xgb.fit(
    X_train_smote,
    y_train_smote
)

xgb_importance = pd.DataFrame({

    "Feature": X_train_smote.columns,

    "XGBoost_Importance": xgb.feature_importances_

})

xgb_importance = xgb_importance.sort_values(

    by="XGBoost_Importance",

    ascending=False

).reset_index(drop=True)

xgb_importance

In [ ]:
xgb_importance.to_csv(

    "Results/Tables/xgboost_feature_importance.csv",

    index=False

)

In [ ]:
plt.figure(figsize=(10,7))

plt.barh(

    xgb_importance["Feature"],

    xgb_importance["XGBoost_Importance"],

    color="#F4C2C2"

)

plt.gca().invert_yaxis()

plt.title(

    "XGBoost Feature Importance",

    fontsize=15,

    fontweight="bold"

)

plt.tight_layout()

plt.savefig(

    "Results/Figures/xgboost_feature_importance.png",

    dpi=300

)

plt.show()

In [ ]:
combined = (
    information_gain_df
    .merge(gain_ratio_df, on="Feature")
    .merge(mi_df, on="Feature")
    .merge(rf_importance, on="Feature")
    .merge(xgb_importance, on="Feature")
)

combined

In [ ]:
combined.to_csv(

    "Results/Tables/combined_feature_selection.csv",

    index=False

)

In [ ]:
combined["IG_Rank"] = combined["Information_Gain"].rank(ascending=False)

combined["GR_Rank"] = combined["Gain_Ratio"].rank(ascending=False)

combined["MI_Rank"] = combined["Mutual_Information"].rank(ascending=False)

combined["RF_Rank"] = combined["RF_Importance"].rank(ascending=False)

combined["XGB_Rank"] = combined["XGBoost_Importance"].rank(ascending=False)

combined["Average_Rank"] = combined[
    [
        "IG_Rank",
        "GR_Rank",
        "MI_Rank",
        "RF_Rank",
        "XGB_Rank"
    ]
].mean(axis=1)

combined = combined.sort_values(
    by="Average_Rank"
)

combined

In [ ]:
combined.to_csv(

    "Results/Tables/final_feature_ranking.csv",

    index=False

)

In [ ]:
# ==========================================
# Keep ALL Features
# ==========================================

top_features = X_train_smote.columns.tolist()

print("Total Features:", len(top_features))
print(top_features)

In [ ]:
# ==========================================
# Create Final Dataset
# ==========================================

X_train_final = X_train_smote[top_features]

X_val_final = X_val_scaled[top_features]

X_test_final = X_test_scaled[top_features]

In [ ]:
import os

os.makedirs("Processed_Data/Final", exist_ok=True)

X_train_final.to_csv(
    "Processed_Data/Final/X_train_final.csv",
    index=False
)

X_val_final.to_csv(
    "Processed_Data/Final/X_val_final.csv",
    index=False
)

X_test_final.to_csv(
    "Processed_Data/Final/X_test_final.csv",
    index=False
)

y_train_smote.to_csv(
    "Processed_Data/Final/y_train_final.csv",
    index=False
)

y_val.to_csv(
    "Processed_Data/Final/y_val_final.csv",
    index=False
)

y_test.to_csv(
    "Processed_Data/Final/y_test_final.csv",
    index=False
)

print("Final dataset saved successfully.")

In [ ]:
print("X_train_final:", X_train_final.shape)
print("X_val_final:", X_val_final.shape)
print("X_test_final:", X_test_final.shape)

print("y_train_final:", y_train_smote.shape)
print("y_val_final:", y_val.shape)
print("y_test_final:", y_test.shape)